# Uso de Redes Neuronales Generativas 

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- Dataset Loader for Variable-Size TSP Instances ---
class CustomTSPDataset(Dataset):
    """
    PyTorch Dataset wrapping a list of coordinate tensors of varying lengths.
    Each tensor is of shape (n_i, 2).
    """
    def __init__(self, coords_list):
        self.coords_list = coords_list

    def __len__(self):
        return len(self.coords_list)

    def __getitem__(self, idx):
        return self.coords_list[idx]


def load_tsp_coords(path: str):
    """
    Load a list of torch_geometric Data objects from `path` and extract their `x` tensors.
    Returns a Python list of tensors of shape (n_i, 2).
    """
    all_data = torch.load(path, weights_only=False)
    coords_list = []
    for d in all_data:
        x = getattr(d, 'x', None)
        if x is None:
            continue
        coords_list.append(x.float())
    if not coords_list:
        raise ValueError(f"No TSP instances found in {path}")
    return coords_list


def collate_fn(batch):
    """
    Collate function that pads each coordinate tensor in `batch` to the maximum length in the batch.
    Returns:
      coords_padded: Tensor (B, N_max, 2)
      lengths: LongTensor (B,) with original lengths
    """
    lengths = torch.tensor([c.size(0) for c in batch], dtype=torch.long)
    N_max = lengths.max().item()
    B = len(batch)
    coords_padded = torch.zeros(B, N_max, 2)
    for i, c in enumerate(batch):
        coords_padded[i, : c.size(0), :] = c
    return coords_padded, lengths


# --- Transformer Pointer Network with Variable-Size Support ---
class TransformerPointer(nn.Module):
    """
    Transformer-based encoder + pointer decoder that handles padded inputs of varying lengths.
    """
    def __init__(self, d_model: int = 128, nhead: int = 8, num_layers: int = 3):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Linear(2, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.project_context = nn.Linear(d_model, d_model)
        self.pointer = nn.Linear(d_model * 2, 1)

    def forward(self, coords: torch.Tensor, lengths: torch.Tensor):
        device = coords.device
        B, N_max, _ = coords.size()
        x = self.embedding(coords)
        pad_mask = torch.arange(N_max, device=device).unsqueeze(0) >= lengths.unsqueeze(1)
        h = self.encoder(x, src_key_padding_mask=pad_mask)

        # Initialize visited mask and context
        visited = pad_mask.clone()  # (B, N_max)
        valid = ~pad_mask
        sum_h = (h * valid.unsqueeze(2)).sum(dim=1)
        context = sum_h / lengths.unsqueeze(1).to(device)

        tours, log_probs = [], []
        for t in range(N_max):
            q = self.project_context(context).unsqueeze(1)
            energy = torch.tanh(q.expand(-1, N_max, -1) + h)
            scores = self.pointer(torch.cat((energy, h), dim=-1)).squeeze(-1)
            scores = scores.masked_fill(visited, float('-inf'))

            idx = torch.zeros(B, dtype=torch.long, device=device)
            log_p = torch.zeros(B, device=device)
            active = lengths > t
            if active.any():
                scores_a = scores[active]
                probs_a = torch.softmax(scores_a, dim=-1)
                sel = torch.multinomial(probs_a, 1).squeeze(-1)
                batch_idx = active.nonzero(as_tuple=True)[0]
                idx = idx.scatter(0, batch_idx, sel)
                lp = torch.log(probs_a.gather(1, sel.unsqueeze(1)).squeeze(1))
                log_p = log_p.scatter(0, batch_idx, lp)

                # Update visited via functional mask
                new_vis = torch.zeros_like(visited)
                new_vis[batch_idx] = visited[active].scatter(1, sel.unsqueeze(1), True)
                visited = visited | new_vis

                # Update context via functional full tensor
                selected = torch.zeros_like(context)
                selected[batch_idx] = h[active, sel, :]
                context = torch.where(active.unsqueeze(1), selected, context)

            tours.append(idx)
            log_probs.append(log_p)

        tours = torch.stack(tours, dim=1)
        log_probs = torch.stack(log_probs, dim=1)
        return tours, log_probs


def compute_reward(coords: torch.Tensor, tours: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    B, N_max, _ = coords.size()
    rewards = []
    for i in range(B):
        n = lengths[i].item()
        tour_i = tours[i, :n]
        pts = coords[i, :n, :]
        ordered = pts[tour_i]
        rolled = torch.cat([ordered[1:], ordered[:1]], dim=0)
        dist = torch.norm(ordered - rolled, dim=1).sum()
        rewards.append(-dist)
    return torch.stack(rewards)


def train_tsp_from_file(dataset_path: str = 'tsps1000.pt', batch_size: int = 128,
                        num_epochs: int = 50, lr: float = 1e-4,
                        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
    coords_list = load_tsp_coords(dataset_path)
    dataset = CustomTSPDataset(coords_list)
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=True, collate_fn=collate_fn)

    model = TransformerPointer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        total_reward = 0.0
        for coords, lengths in loader:
            coords = coords.to(device)
            lengths = lengths.to(device)
            tours, log_probs = model(coords, lengths)
            rewards = compute_reward(coords, tours, lengths)

            loss = -(log_probs.sum(dim=1) * rewards.detach()).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * coords.size(0)
            total_reward += rewards.sum().item()

        avg_loss = total_loss / len(dataset)
        avg_rew = total_reward / len(dataset)
        print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Avg Reward: {avg_rew:.4f}")

    torch.save(model.state_dict(), 'tsp_transformer_varlen.pth')
    return model


if __name__ == '__main__':
    train_tsp_from_file(dataset_path='tsps1000.pt', batch_size=64,
                        num_epochs=100, lr=5e-4)


Epoch 01 | Loss: -8307.1034 | Avg Reward: -507.8895
Epoch 02 | Loss: -5955.5708 | Avg Reward: -474.0647
Epoch 03 | Loss: -6737.0124 | Avg Reward: -487.4409
Epoch 04 | Loss: -8791.1390 | Avg Reward: -520.9736
Epoch 05 | Loss: -10760.1850 | Avg Reward: -546.0536
Epoch 06 | Loss: -10756.4489 | Avg Reward: -548.1385
Epoch 07 | Loss: -10711.1698 | Avg Reward: -547.8751
Epoch 08 | Loss: -10570.0475 | Avg Reward: -548.1210
Epoch 09 | Loss: -10548.2913 | Avg Reward: -548.2246
Epoch 10 | Loss: -10210.9280 | Avg Reward: -543.1941
Epoch 11 | Loss: -10582.7100 | Avg Reward: -548.4855
Epoch 12 | Loss: -10259.4930 | Avg Reward: -547.1554


KeyboardInterrupt: 